# Assignment 1

Deadline: 19.03.2026, 12:00 CET

<Add your name, student-id and emal address>

In [3]:
# Import standard libraries
import os
import sys
import timeit # To compute runtimes
from typing import Optional
from pathlib import Path

# Import third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import local modules — robust path setup to avoid ModuleNotFoundError
def find_repo_root(max_levels=6):
    p = Path.cwd()
    for _ in range(max_levels):
        if (p / 'src').exists():
            return str(p)
        p = p.parent
    return str(Path.cwd())

project_root = os.path.dirname(os.getcwd())   # Change this path if needed
print(f"Project root found at: {project_root}")
print(f"Current working directory: {os.getcwd()}")
src_path = os.path.join(project_root, "src")
# Put src first so imports like `from estimation...` work
if src_path not in sys.path:
    sys.path.insert(0, src_path)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from estimation.covariance import Covariance, is_pos_def, make_pos_def
from estimation.expected_return import ExpectedReturn
from optimization.constraints import Constraints
from optimization.optimization import Optimization, Objective
from optimization.optimization_data import OptimizationData
from optimization.quadratic_program import QuadraticProgram, USABLE_SOLVERS
from helper_functions import simulate_correlated_gbm

Project root found at: /Users/nawangpegentsang/Desktop/qpmwp-course
Current working directory: /Users/nawangpegentsang/Desktop/qpmwp-course/assignments


## 1. Solver Horse Race

### 1.a)
(3 points)

Generate a synthetic dataset of dimension TxN, T=1000, N=50, and compute a vector of expected returns, q, and a covariance matrix, P, using classes ExpectedReturn and Covariance respectively.

In [ ]:

# Set the dimensions
T = 1000       # Number of time steps
N = 50         # Number of assets
rnd_seed = 42  # Random seed for reproducibility

# Set random seed for reproducibility
np.random.seed(rnd_seed)

# Generate a random mean vector (annualised drifts between 0% and 20%)
mu = np.random.uniform(0.0, 0.20, size=N)

# Generate a random positive-definite covariance matrix
# Step 1: draw a random matrix and form a symmetric PD matrix via A @ A.T
A = np.random.randn(N, N)
sigma_raw = A @ A.T / N          # scale down so eigenvalues are reasonable
# Step 2: ensure positive-definiteness (should already be PD, but apply
#         make_pos_def as a safety net)
sigma = make_pos_def(sigma_raw) if not is_pos_def(sigma_raw) else sigma_raw

# Generate correlated geometric Brownian motion paths and compute discrete returns
prices = simulate_correlated_gbm(mu=mu, sigma=sigma, T=T, random_seed=None)
returns = prices.pct_change().dropna()

# Compute the vector of expected returns from the return series using ExpectedReturn
er = ExpectedReturn(method='geometric')   # default: geometric mean
er.estimate(X=returns)                    # stores result in er.vector
q = er.vector                             # pd.Series, shape (N,)

# Compute the covariance matrix from the return series using Covariance
cov = Covariance(method='pearson', check_positive_definite=True)
cov.estimate(X=returns)                   # stores result in cov.matrix
P = cov.matrix                            # pd.DataFrame, shape (N, N)

# Display the results
print("Vector of expected returns (q):")
print(q)

print("\nCovariance matrix (P):")
print(P)

NameError: name 'mu' is not defined

### 1.b)
(3 points)

Instantiate a constraints object by injecting column names of the return series created in 1.a) as ids and add:
- a budget constaint (i.e., asset weights have to sum to one)
- lower bounds of 0.0 for all assets
- upper bounds of 0.2 for all assets
- group contraints such that the sum of the weights of the first 15 assets is <= 0.3, the sum of assets 16 to 45 is <= 0.4 and the sum of assets 41 to 50 is <= 0.5

In [ ]:
# Instantiate the Constraints class using column names of returns as ids
constraints = Constraints(ids=returns.columns.tolist())

# Add budget constraint: sum of weights = 1
constraints.add_budget(rhs=1, sense='=')

# Add box constraints: lower = 0.0, upper = 0.2 for every asset
constraints.add_box(box_type='LongOnly', lower=0.0, upper=0.2)

# Add linear (group) constraints
ids = returns.columns.tolist()

# Group 1: first 15 assets (indices 0–14)  <=  0.3
g1 = pd.Series(0.0, index=ids)
g1.iloc[0:15] = 1.0
constraints.add_linear(g_values=g1, sense='<=', rhs=0.3, name='group1')

# Group 2: assets 16 to 45 (indices 15–44)  <=  0.4
g2 = pd.Series(0.0, index=ids)
g2.iloc[15:45] = 1.0
constraints.add_linear(g_values=g2, sense='<=', rhs=0.4, name='group2')

# Group 3: assets 41 to 50 (indices 40–49)  <=  0.5
g3 = pd.Series(0.0, index=ids)
g3.iloc[40:50] = 1.0
constraints.add_linear(g_values=g3, sense='<=', rhs=0.5, name='group3')

# Display selected columns of the G matrix to verify the group constraints
constraints.linear['G'][['Asset_1', 'Asset_15', 'Asset_16', 'Asset_40', 'Asset_41', 'Asset_50']]

### 1.c) 
(4 points)

Solve a Mean-Variance optimization problem (using coefficients P and q in the objective function) which satisfies the above defined constraints.
Repeat the task for all open-source solvers in qpsolvers that you could install and compare the results in terms of:

- runtime
- accuracy: value of the primal problem.
- reliability: are all constraints fulfilled? Extract primal residuals, dual residuals and duality gap.

Generate a DataFrame with the solvers as column names and the following row index: 'solution_found': bool, 'objective': float, 'primal_residual': float, 'dual_residual': float, 'duality_gap': float, 'runtime': float.

Put NA's for solvers where the optimization failed for some reason.




In [ ]:
# Extract the constraints in the format required by the solver
GhAb = constraints.to_GhAb()
lb = constraints.box['lower'].to_numpy()
ub = constraints.box['upper'].to_numpy()

# Convert P and q to numpy arrays
P_np = P.to_numpy() if isinstance(P, pd.DataFrame) else P
q_np = q.to_numpy() if isinstance(q, pd.Series) else q

# Objective for Mean-Variance: minimise  risk_aversion * w' P w  - q' w
# Using risk_aversion = 1 (standard MV), matching MeanVariance.set_objective:
#   P_obj = 2 * risk_aversion * P,   q_obj = -q
risk_aversion = 1
P_obj = 2 * risk_aversion * P_np
q_obj = -q_np

# Define a dictionary to store the results in case a solver fails
result_on_fail = {
    'solution_found': False,
    'objective': np.nan,
    'primal_residual': np.nan,
    'dual_residual': np.nan,
    'duality_gap': np.nan,
    'runtime': np.nan,
}

results_dict = {}

for solver in USABLE_SOLVERS:
    try:
        qp = QuadraticProgram(
            P=P_obj.copy(),
            q=q_obj.copy(),
            G=GhAb['G'],
            h=GhAb['h'],
            A=GhAb['A'],
            b=GhAb['b'],
            lb=lb,
            ub=ub,
            solver=solver,
        )

        # Time the solve call
        start = timeit.default_timer()
        qp.solve()
        elapsed = timeit.default_timer() - start

        sol = qp.results['solution']

        if sol.found and sol.x is not None:
            obj_val = qp.objective_value(x=sol.x, constant=False)
            results_dict[solver] = {
                'solution_found': True,
                'objective': obj_val,
                'primal_residual': sol.primal_residual() if callable(getattr(sol, 'primal_residual', None)) else getattr(sol, 'primal_residual', np.nan),
                'dual_residual': sol.dual_residual() if callable(getattr(sol, 'dual_residual', None)) else getattr(sol, 'dual_residual', np.nan),
                'duality_gap': sol.duality_gap() if callable(getattr(sol, 'duality_gap', None)) else getattr(sol, 'duality_gap', np.nan),
                'runtime': elapsed,
            }
        else:
            results_dict[solver] = dict(result_on_fail)
            results_dict[solver]['runtime'] = elapsed

    except Exception as e:
        print(f"Solver '{solver}' failed with error: {e}")
        results_dict[solver] = dict(result_on_fail)

# Build the summary DataFrame
index_order = ['solution_found', 'objective', 'primal_residual', 'dual_residual', 'duality_gap', 'runtime']
results_df = pd.DataFrame(results_dict).loc[index_order]

print(results_df)

Print and visualize the results

In [ ]:
# ── Pretty-print the results table ──────────────────────────────────────────
print("=" * 70)
print("Solver comparison – Mean-Variance optimisation (N=50, T=1000)")
print("=" * 70)
print(results_df.to_string())

# ── Filter to solvers that found a solution ──────────────────────────────────
successful = results_df.loc[:, results_df.loc['solution_found'] == True].copy()
numeric_rows = ['objective', 'primal_residual', 'dual_residual', 'duality_gap', 'runtime']
successful_num = successful.loc[numeric_rows].astype(float)

# ── Figure 1: runtime bar chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Runtime
runtimes = successful_num.loc['runtime'].sort_values()
axes[0].bar(runtimes.index, runtimes.values, color='steelblue')
axes[0].set_title('Runtime (seconds)')
axes[0].set_ylabel('seconds')
axes[0].set_xlabel('Solver')
axes[0].tick_params(axis='x', rotation=45)

# Objective value (accuracy)
obj_vals = successful_num.loc['objective'].sort_values()
axes[1].bar(obj_vals.index, obj_vals.values, color='darkorange')
axes[1].set_title('Objective value (lower = better)')
axes[1].set_ylabel('Objective')
axes[1].set_xlabel('Solver')
axes[1].tick_params(axis='x', rotation=45)

# Primal residual (reliability)
prim_res = successful_num.loc['primal_residual'].sort_values()
axes[2].bar(prim_res.index, prim_res.values, color='seagreen')
axes[2].set_title('Primal residual (lower = more reliable)')
axes[2].set_ylabel('Primal residual')
axes[2].set_xlabel('Solver')
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Solver Horse Race – Mean-Variance Optimisation', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2. Analytical Solution to Minimum-Variance Problem

(5 points)

- Create a `MinVariance` class that follows the structure of the `MeanVariance` class.
- Implement the `solve` method in `MinVariance` such that if `solver_name = 'analytical'`, the analytical solution is computed and stored within the object (if such a solution exists). If not, call the `solve` method from the parent class.
- Create a `Constraints` object by injecting the same ids as in part 1.b) and add a budget constraint.
- Instantiate a `MinVariance` object by setting `solver_name = 'analytical'` and passing instances of `Constraints` and `Covariance` as arguments.
- Create an `OptimizationData` object that contains an element `return_series`, which consists of the synthetic data generated in part 1.a).
- Solve the optimization problem using the created `MinVariance` object and compare the results to those obtained in part 1.c).


In [ ]:
# ── Analytical minimum-variance solution (budget constraint only) ────────────
#
# Problem:  min  w' Σ w
#           s.t. 1' w = 1
#
# Closed-form solution via Lagrange multipliers:
#
#   w* = Σ^{-1} 1 / (1' Σ^{-1} 1)
#
# where 1 is the vector of ones.

from optimization.optimization import Optimization, Objective

class MinVariance(Optimization):

    def __init__(self,
                 constraints: Constraints,
                 covariance: Optional[Covariance] = None,
                 **kwargs):
        super().__init__(
            constraints=constraints,
            **kwargs
        )
        self.covariance = Covariance() if covariance is None else covariance

    def set_objective(self, optimization_data: OptimizationData) -> None:
        """Set the QP objective for the minimum-variance problem:
           min  w' Σ w   →   P = 2Σ,  q = 0
        """
        X = optimization_data['return_series']
        covmat = self.covariance.estimate(X=X, inplace=False)
        n = X.shape[1]
        self.objective = Objective(
            P=covmat * 2,          # factor 2 because QP convention is 0.5 x'Px
            q=np.zeros(n),
        )
        return None

    def solve(self) -> None:
        if self.params.get('solver_name') == 'analytical':
            # ── Analytical solution ──────────────────────────────────────────
            # Requires P (covariance matrix) to be stored in the objective.
            # P_obj = 2 * Sigma  →  Sigma = P_obj / 2
            P_obj = self.objective.coefficients.get('P')
            if P_obj is None:
                raise ValueError("Call set_objective() before solve().")

            # Recover Sigma from the QP coefficient
            if isinstance(P_obj, pd.DataFrame):
                Sigma = P_obj.to_numpy() / 2.0
                asset_ids = P_obj.columns.tolist()
            else:
                Sigma = np.array(P_obj) / 2.0
                asset_ids = self.constraints.ids

            n = Sigma.shape[0]
            ones = np.ones(n)

            # Invert the covariance matrix
            try:
                Sigma_inv = np.linalg.inv(Sigma)
            except np.linalg.LinAlgError:
                # Fallback: use pseudo-inverse
                Sigma_inv = np.linalg.pinv(Sigma)

            # w* = Σ^{-1} 1 / (1' Σ^{-1} 1)
            num = Sigma_inv @ ones
            denom = ones @ Sigma_inv @ ones

            if abs(denom) < 1e-12:
                raise ValueError("Degenerate covariance matrix: cannot compute analytical solution.")

            w_star = num / denom

            weights = pd.Series(w_star, index=asset_ids)
            obj_val = float(w_star @ Sigma @ w_star)

            self.results.update({
                'weights': weights.to_dict(),
                'status': True,
                'objective': obj_val,
            })
            return None
        else:
            return super().solve()


# ── Create Constraints with budget constraint only ───────────────────────────
constraints_mv = Constraints(ids=returns.columns.tolist())
constraints_mv.add_budget(rhs=1, sense='=')

# ── Instantiate MinVariance with analytical solver ───────────────────────────
cov_mv = Covariance(method='pearson', check_positive_definite=True)
min_var = MinVariance(
    constraints=constraints_mv,
    covariance=cov_mv,
    solver_name='analytical',
)

# ── Prepare OptimizationData ─────────────────────────────────────────────────
opt_data = OptimizationData(return_series=returns)

# ── Set objective (estimates the covariance matrix internally) ───────────────
min_var.set_objective(optimization_data=opt_data)

# ── Solve analytically ───────────────────────────────────────────────────────
min_var.solve()

# ── Display weights ───────────────────────────────────────────────────────────
w_analytical = pd.Series(min_var.results['weights'])
print("Analytical minimum-variance weights:")
print(w_analytical.round(6))
print(f"\nSum of weights : {w_analytical.sum():.6f}  (should be 1.0)")
print(f"Min weight     : {w_analytical.min():.6f}")
print(f"Max weight     : {w_analytical.max():.6f}")
print(f"Portfolio variance (objective): {min_var.results['objective']:.8f}")